In [ ]:
cosmos_db_url = "https://csdb-intertek-esus-dev.documents.azure.com:443/"
cosmos_db_key = "azcUeVxFxoYoFkChvWI8Wr8lMijOuWXDYQsvMf6O2LmT0Uv3Zs7lDPiXSxWYOjq00MFDbK88ApotACDbODLFXA=="
cosmos_db_database = "intertek_poc_dev"
cosmos_db_project_container = "projects"


COSMOS_DB_URI = cosmos_db_url
COSMOS_DB_KEY = cosmos_db_key
COSMOS_DB_DATABASE = cosmos_db_database
COSMOS_PROJECT_CONTAINER = cosmos_db_project_container

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey


"""
Configure Cosmos DB partitioning using '/id' for scalable data distribution.
Input values used to identify and query the target project document.
Create an authenticated Cosmos DB client connection.
Connect to the required database and container holding project data.
Define a parameterized query to safely fetch the project by Project_Id.
Execute the query across partitions and collect matching documents.
Validate that the project exists before continuing.
Display the matching project documents.


Checks:
    1. If the client is able to access Azure Cosmos DB.
    2. If Azure Cosmos DB container, partition, and project exists and is able to be accessed.
"""


# Container partition configuration
cosmos_container_properties = {"partition_key": PartitionKey(path="/id")}

project_id = "G15006730"
first_name = "test_user9"

# Create Cosmos DB client
cosmos_client = CosmosClient(COSMOS_DB_URI, credential=COSMOS_DB_KEY)

# Connect to database and container
db = cosmos_client.get_database_client(COSMOS_DB_DATABASE)
projects_container = db.get_container_client(COSMOS_PROJECT_CONTAINER)

# Query project by Project_Id
query = """
SELECT *
FROM c
WHERE c.Project_Id = @project_id
"""

parameters = [{"name": "@project_id", "value": project_id}]

# Run query across partitions
docs = list(
    projects_container.query_items(
        query=query,
        parameters=parameters,
        enable_cross_partition_query=True,
    )
)

# Ensure project exists
if not docs:
    raise Exception(f"Project '{project_id}' not found")

print(docs)

[{'id': '1a9d59e1-59a7-4a2f-b898-46ee2a18dbed', 'Project_Id': 'G15006730', 'Standard': 'IEC_61010-1', 'Client_Name': 'Client 1', 'Product': 'Product 1', 'Source_Doc': [{'filename': 'Blockdiagram.pdf', 'url': 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/Blockdiagram.pdf', 'uploaded_at': '2026-01-28T07:42:04.447630'}, {'filename': 'BM-2310221-3A.pdf', 'url': 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-3A.pdf', 'uploaded_at': '2026-01-28T07:42:04.447682'}, {'filename': 'BM-2310221-3B.pdf', 'url': 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-3B.pdf', 'uploaded_at': '2026-01-28T07:42:04.447693'}, {'filename': 'BM-2310221-04.pdf', 'url': 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-04.pdf', 'uploaded_at': '

In [ ]:
"""
Checks:
    1. If blob URLs are present in the docs
"""


# Get the first project document from the collection
project_doc = docs[0]

# Extract source document metadata safely
source_docs = project_doc.get("Source_Doc") or []


def extract_blob_urls(source_documents):
    """
    Extract blob storage URLs from source document metadata.

    This function collects all valid URLs from the source documents
    so they can be used for downstream processing such as file access,
    validation, or ingestion.
    """
    blob_urls = [
        doc["url"] for doc in source_documents if isinstance(doc, dict) and "url" in doc
    ]

    print(f"Extracted {len(blob_urls)} blob URLs")

    return blob_urls


# Generate list of blob URLs
blob_urls = extract_blob_urls(source_docs)

print(blob_urls)


Extracted 15 blob URLs
['https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/Blockdiagram.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-3A.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-3B.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-04.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-05.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BOM-PR0006730-Client.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BOM-PR0006730-IO.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusd

In [4]:
"""
Checks:
    1. Vector store naming convention
        a. vectorstorecontainer_new_itk_text_{first_name}_{project_id}
        b. vectorstorecontainer_new_itk_image_{first_name}_{project_id}
"""


# User and project identifiers
first_name = "Yogesh"
project_id = "G1050004"


def generate_container_names(user_name, project_identifier):
    """
    Generate Azure storage container names for text and image vector stores.

    These container names are used to organize and separate project-specific
    text and image embeddings within the storage system.
    """
    text_container = (
        f"vectorstorecontainer_new_itk_text_{user_name}_{project_identifier}"
    )

    image_container = (
        f"vectorstorecontainer_new_itk_image_{user_name}_{project_identifier}"
    )

    print(f"Image container: {image_container}")
    print(f"Text container: {text_container}")

    return text_container, image_container


# Create container names
text_container, image_container = generate_container_names(
    first_name,
    project_id,
)

Image container: vectorstorecontainer_new_itk_image_Yogesh_G1050004
Text container: vectorstorecontainer_new_itk_text_Yogesh_G1050004


In [ ]:
from pathlib import Path
from utility.embeddings import ingest_files_from_blob_urls_create_embeddings

"""
========================================
        INGESTION PIPELINE WORKFLOW
========================================

1. Initialize the ingestion pipeline with:
   - Project ID
   - Blob URLs
   - Download paths
   - Cosmos DB container name
2. Create a connection to Azure Cosmos DB using `CosmosClient`
3. Create or fetch the text vector container with vector indexing configuration
4. Delete all existing documents from the text vector container for fresh ingestion
5. Create local working directories for:
   - Source files (`src_files`)
   - Extracted images (`image_files`)
6. Download all files from Azure Blob Storage URLs into the local `src_files` directory.
7. Convert DOCX files into PDFs and store them in the same working directory.
8. Collect all downloaded and converted PDF paths into a single processing list.
9. Detect editable PDFs (such as CIS forms) using `extract_cis()`.
10. Extract editable PDF pages as images and store them in the `image_files` directory.
11. Remove editable PDFs from the normal PDF text ingestion pipeline.
12. Copy extracted editable PDF images into the source working directory.
13. Clean extracted text content to remove formatting and OCR artifacts.
14. Merge CIS extracted text into the main extracted text collection.
15. Load all PDFs and extract text content page-by-page.
16. Split extracted PDF text into chunks using configured:
    - Chunk size
    - Chunk overlap
17. Detect CAD/schematic pages and generate image metadata for them.
18. Append editable PDF images into the CAD/schematic image metadata list.
19. Clear the text Cosmos DB container again before vector ingestion.
20. Build the text vector store wrapper connected to Cosmos DB.
21. Assign UUIDs to all text chunks for unique vector document IDs.
22. Generate embeddings for text chunks using Azure OpenAI embeddings.
23. Ingest all text chunk embeddings into Cosmos DB in parallel batches.
24. Deduplicate extracted page images using local image paths.
25. Upload extracted CAD/schematic page images to Azure Blob Storage.
26. Generate blob URLs for all uploaded page images.
27. Flatten nested image URL structures into a single list of image URLs.
28. Send all extracted images to the GPT-4 Vision OCR pipeline.
29. Extract OCR text and image understanding results from CAD/schematic images.
30. Create or fetch the image vector Cosmos DB container.
31. Clear the image vector container before fresh image ingestion.
32. Build the image vector store wrapper connected to Cosmos DB.
33. Assign UUIDs to all OCR/image documents.
34. Generate embeddings for OCR/image text content.
35. Ingest all image embeddings into Cosmos DB in parallel batches.
36. Complete the ingestion pipeline after successful:
    - Text vector storage
    - Image vector storage
37. Return final ingestion metadata including:
    - Project ID
    - Image URLs
    - PDF paths
    - Image metadata
    - Chunk count

========================================
         END OF INGESTION FLOW
========================================
"""

BASE_DIR = Path.cwd()
BASE_DIR
DOWNLOAD_DIR = BASE_DIR / "data" / "src_files"
ingest_files_from_blob_urls_create_embeddings(
    DOWNLOAD_DIR, blob_urls, project_id, text_container, image_container
)

d:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\.venv\Lib\site-packages\azure\search\documents\indexes\_generated\models\_models_py3.py:5644: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
  pattern: str = "\W+",
d:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\.venv\Lib\site-packages\azure\search\documents\indexes\_generated\models\_models_py3.py:5869: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
  pattern: str = "\W+",


Running in LOCAL mode – using .env
AOAI_ENDPOINT https://oai-intertek-esus2-dev.openai.azure.com/
######### download_dir ######## d:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\data\src_files
######## blob_urls ######### ['https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/Blockdiagram.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-3A.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-3B.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-04.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BM-2310221-05.pdf', 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/BOM-PR0006

100%|██████████| 1/1 [00:04<00:00,  4.80s/it]


[INFO] Converted DOCX to PDF: D:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\data\G1050004\src_files\CyberLogiQ Controller Manual 2024 ETL.docx -> D:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\data\G1050004\src_files\CyberLogiQ Controller Manual 2024 ETL.pdf
[INFO] Processing URL #10: https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/CyberlogiQ%20Controller%20Technical%20Specifications%20rev.%201.pdf
[INFO] Downloaded PDF recorded: D:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\data\G1050004\src_files\CyberlogiQ Controller Technical Specifications rev. 1.pdf
[INFO] Processing URL #11: https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G15006730/source_documents/PO_for_Qu-01414060-2_Intertek_PPC0010377.pdf
[INFO] Downloaded PDF recorded: D:\code\intertek\InterTek-AI-Repo_dev_Akshay_latest 4\Backend\data\G1050004\src_files\PO_for_Qu-01414060-2_Intertek_PPC

{'project_id': 'G1050004',
 'image_urls': ['https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G1050004/pdf_images/Blockdiagram.pdf/page_2.png',
  'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G1050004/pdf_images/BOM-PR0006730-IO.pdf/page_1.png',
  'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G1050004/pdf_images/Blockdiagram.pdf/page_1.png',
  'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G1050004/pdf_images/BM-2310221-05.pdf/page_1.png',
  'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G1050004/pdf_images/BM-2310221-05.pdf/page_2.png',
  'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G1050004/pdf_images/BOM-PR0006730-Client.pdf/page_1.png',
  'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/Documents/G1050004/pdf_images/BM-2310221-04.pdf/page_2.png',
  'htt

In [ ]:
import json
from pathlib import Path

BASE_DIR = Path.cwd()
BASE_DIR
DOWNLOAD_DIR = BASE_DIR / "data" / "src_files"

# --------------------------------------------------------
# Input JSON configuration
# --------------------------------------------------------

# Base directory containing project data
DATA_DIR = BASE_DIR / "data"

# List of input JSON files to process
INPUT_JSON_FILENAMES = [
    "pta_final_6_3_1_part1.json",
    "pta_final_6_3_1_part2.json",
    "pta_final_6_3_1_part3.json",
    "pta_final_6_3_1_part4.json",
    "pta_final_6_3_1_part5.json",
]

# Generate full file paths for all input JSON files
INPUT_JSONS = [DATA_DIR / "input_files" / filename for filename in INPUT_JSON_FILENAMES]

print("Input JSON files loaded:")
for path in INPUT_JSONS:
    print(path)


def load_json_file(file_path):
    """
    Load and parse a JSON file.

    This helper function centralizes JSON loading so file
    processing remains clean and reusable across workflows.
    """
    print(f"\nLoading JSON file: {file_path}")

    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)


def normalize_image_urls(image_urls):
    """
    Normalize image metadata into a consistent structure.

    This normalization ensures downstream task generation
    works uniformly regardless of the original image format.
    """
    normalized_images = []

    for image in image_urls:
        if isinstance(image, dict):
            image_file = (image.get("image_file") or "").split("?")[0]

            normalized_images.append(
                {
                    "pdf_file": image.get("pdf_file"),
                    "page": image.get("page"),
                    "image_file": image_file,
                    "url": image.get("url"),
                }
            )

        else:
            clean_file_name = image.split("/")[-1].split("?")[0]

            normalized_images.append(
                {
                    "pdf_file": None,
                    "page": None,
                    "image_file": clean_file_name,
                    "url": image,
                }
            )

    print(f"Normalized {len(normalized_images)} image URLs")

    return normalized_images


def build_tasks_with_custom_prompt_grey(data, image_urls):
    """
    Build AI extraction tasks from table item definitions.

    This function prepares structured task payloads for AI-driven
    extraction workflows while preserving metadata such as task type,
    accuracy settings, and custom prompts.
    """
    tasks = []
    item_refs = []

    # Normalize image metadata before task generation
    normalized_images = normalize_image_urls(image_urls)

    # Process all table items
    for table in data.get("Tables", []):
        for item in table.get("Items", []):
            # Skip non-AI-fillable items
            if item.get("ai_fillable") is not True:
                continue

            custom_prompt = item.get("custom_prompt", "")
            accuracy_level = bool(item.get("accuracy_level", False))

            # Default grey flag behavior
            grey_flag = bool(item.get("grey")) if "grey" in item else not accuracy_level

            task_type = str(item.get("task_type", "extract")).strip().lower()

            field = item.get("field") or ""

            # Generate question prompt
            if isinstance(custom_prompt, str) and custom_prompt.strip():
                modified_question = custom_prompt.strip()

            elif task_type == "verdict":
                modified_question = (
                    f"Give the verdict as Pass, Fail, or N/A only.\n{field}"
                )

            else:
                modified_question = field

            # Create task payload
            tasks.append(
                {
                    "question": modified_question,
                    "image": list(normalized_images),
                    "task_type": task_type,
                    "custom_prompt": custom_prompt,
                    "accuracy_level": accuracy_level,
                    "grey": grey_flag,
                }
            )

            item_refs.append(item)

    print(f"Generated {len(tasks)} AI tasks")

    return tasks, item_refs


# --------------------------------------------------------
# STEP 2 — BUILD TASKS
# --------------------------------------------------------

print("\n===============================================")
print("           STEP 2 — BUILD TASKS")
print("===============================================")

# Load and process each input JSON file
for input_json_path in INPUT_JSONS:
    data = load_json_file(input_json_path)

# Build tasks from the final loaded dataset
tasks, item_refs = build_tasks_with_custom_prompt_grey(
    data,
    blob_urls,
)

print(f"\nTask object type: {type(tasks)}")
print(f"Total task references: {len(item_refs)}")

# Preview first task and item reference
print("\nFirst task preview:")
print(tasks[0])

print("\nFirst item reference preview:")
print(item_refs[0])

Input JSON files loaded:
d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part1.json
d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part2.json
d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part3.json
d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part4.json
d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part5.json

           STEP 2 — BUILD TASKS

Loading JSON file: d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part1.json

Loading JSON file: d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part2.json

Loading JSON file: d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part3.json

Loading JSON file: d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part4.json

Loading JSON file: d:\code\InterTek-AI-Repo\Backend\data\input_files\pta_final_6_3_1_part5.json
Normalized 15 image URLs
Generated 153 AI tasks

Task object type: <clas

In [ ]:
# --------------------------------------------------------
# Import vector store and RAG utilities
# --------------------------------------------------------

from utility.trf_report.trf_generation import (
    build_vectorstore_text,
    build_vectorstore_image,
    update_tasks_with_top5_images,
)

# --------------------------------------------------------
# Build vector stores
# --------------------------------------------------------


def initialize_vectorstores(text_container_name, image_container_name):
    """
    Initialize text and image vector stores for retrieval workflows.

    These vector stores enable semantic search across project text
    and image embeddings for downstream RAG-based processing.
    """
    print("Building text vector store...")
    text_vectorstore = build_vectorstore_text(text_container_name)

    print("Building image vector store...")
    image_vectorstore = build_vectorstore_image(image_container_name)

    print("Vector stores initialized successfully")

    return text_vectorstore, image_vectorstore


# Create vector stores
vs, vs2 = initialize_vectorstores(
    text_container,
    image_container,
)

# --------------------------------------------------------
# Configure retrievers
# --------------------------------------------------------


def create_retrievers(text_vectorstore, image_vectorstore, top_k=5):
    """
    Create retriever objects for semantic similarity search.

    Retrievers are used to fetch the most relevant text chunks
    and images during AI task execution.
    """
    print(f"Creating retrievers with top-{top_k} search")

    text_retriever = text_vectorstore.as_retriever(search_kwargs={"k": top_k})

    image_retriever = image_vectorstore.as_retriever(search_kwargs={"k": top_k})

    return text_retriever, image_retriever


# Initialize retrievers
retriever, image_retriever_agent = create_retrievers(
    vs,
    vs2,
)

# --------------------------------------------------------
# Update tasks with relevant images
# --------------------------------------------------------


def enrich_tasks_with_images(task_list, image_retriever):
    """
    Attach the top relevant images to each AI task.

    This enrichment step improves multimodal context retrieval
    by linking tasks with semantically related images.
    """
    print("Updating tasks with top matching images...")

    updated_tasks = update_tasks_with_top5_images(
        task_list,
        image_retriever,
    )

    print(f"Updated {len(updated_tasks)} tasks with image references")

    return updated_tasks


# Enrich tasks using image retrieval
result = enrich_tasks_with_images(
    tasks,
    image_retriever_agent,
)

# Preview first enriched task
print("\nFirst enriched task preview:")
print(result[0])

Running in LOCAL mode – using .env
Building text vector store...
Building image vector store...
Vector stores initialized successfully
Creating retrievers with top-5 search
Updating tasks with top matching images...
[START] Processing 153 tasks with 10 parallel threads.

[INFO] Processing task 1/153 → Verify if the manufacturer mentioned a fluid anywhere in the documentation, the equipment must be safe when exposed to that fluid.Mention 'No fluid considered' incase there is no mention of any fluids by the manufacturer in the documentation.
[INFO] Processing task 2/153 → Verify if the manufacturer mentioned a fluid anywhere in the documentation, the equipment must be safe when exposed to that fluid.
[INFO] Processing task 3/153 → Verify incase there are battery electrolytes the leakage presents no hazard.
[INFO] Processing task 4/153 → Verify and mention the degree of Ingress Protection (IP code) rated for the equipment for solid foreign objects and liquids as per IEC 60529.
[INFO] Proc

In [41]:
# --------------------------------------------------------
# Import Azure OpenAI and RAG utilities
# --------------------------------------------------------

from langchain_openai import AzureChatOpenAI
from langchain_community.callbacks import get_openai_callback

from utility.trf_report.trf_generation import (
    AOAI_ENDPOINT,
    AOAI_KEY,
    API_VERSION,
    CHAT_DEPLOY,
    build_rag_image_pipeline_grey,
    build_vision_message_grey,
    attach_supporting_refs_grey,
)

# --------------------------------------------------------
# Initialize Azure OpenAI LLM
# --------------------------------------------------------


def initialize_llm():
    """
    Create and configure the Azure OpenAI chat model.

    This LLM is used for structured AI extraction and response
    generation within the RAG processing pipeline.
    """
    print("Initializing Azure OpenAI model...")

    llm_instance = AzureChatOpenAI(
        azure_endpoint=AOAI_ENDPOINT,
        api_key=AOAI_KEY,
        openai_api_version="2025-01-01-preview",
        azure_deployment="gpt-5.4",
        temperature=0.1,
    ).with_config({"response_format": "json_object"})

    print("LLM initialized successfully")

    return llm_instance


# Create LLM instance
llm = initialize_llm()

# --------------------------------------------------------
# Build RAG image pipeline
# --------------------------------------------------------


def initialize_rag_pipeline(
    retriever_instance,
    llm_instance,
    vectorstore,
):
    """
    Build the multimodal RAG pipeline for task execution.

    This pipeline combines semantic retrieval, image context,
    and LLM reasoning to generate structured AI responses.
    """
    print("Building RAG image pipeline...")

    rag_pipeline = build_rag_image_pipeline_grey(
        retriever_instance,
        llm_instance,
        build_vision_message_grey,
        attach_supporting_refs_grey,
        vectorstore,
    )

    print("RAG image pipeline initialized")

    return rag_pipeline


# Create RAG pipeline
rag_image = initialize_rag_pipeline(
    retriever,
    llm,
    vs,
)

print(rag_image)

# --------------------------------------------------------
# Run single task with token statistics
# --------------------------------------------------------


def run_single_task_stats(task, rag_pipeline):
    """
    Execute a single AI task and capture token usage statistics.

    This function helps monitor prompt, completion, and total
    token consumption for performance and cost tracking.
    """
    print("Running task through RAG pipeline...")

    with get_openai_callback() as callback:
        response = rag_pipeline.invoke(task)

    # Attach token usage metadata
    response["_token_usage"] = {
        "prompt": callback.prompt_tokens,
        "completion": callback.completion_tokens,
        "total": callback.total_tokens,
    }

    print("Task completed successfully")
    print(f"Total tokens used: {callback.total_tokens}")

    return response

Initializing Azure OpenAI model...
LLM initialized successfully
Building RAG image pipeline...
RAG image pipeline initialized
first={
  retrieved_docs: RunnableLambda(itemgetter('question'))
                  | AzureCosmosDBNoSqlVectorStoreRetriever(vectorstore=<langchain_azure_ai.vectorstores.azure_cosmos_db_no_sql.AzureCosmosDBNoSqlVectorSearch object at 0x00000259235D4EC0>, search_kwargs={'with_embedding': False, 'offset_limit': None, 'projection_mapping': None, 'full_text_rank_filter': None, 'where': None, 'weights': None, 'score_threshold': 0.5}),
  question: RunnableLambda(itemgetter('question')),
  image: RunnableLambda(itemgetter('image')),
  task_type: RunnableLambda(itemgetter('task_type')),
  custom_prompt: RunnableLambda(itemgetter('custom_prompt')),
  accuracy_level: RunnableLambda(itemgetter('accuracy_level')),
  grey: RunnableLambda(itemgetter('grey'))
} middle=[{
  inputs: RunnableLambda(lambda x: {**x, 'context': extract_docs(x.get('retrieved_docs')), 'retrieved_docs':

In [42]:
def fetch_task_by_custom_prompt(tasks, target_prompt):
    """
    Fetch a task from tasks list matching the given custom_prompt.
    Matching strategy:
    1. Exact match
    2. Normalized (case + whitespace) match
    """

    if not target_prompt:
        return None

    # -----------------------------
    # 1. Exact match (fast path)
    # -----------------------------
    for task in tasks:
        if task.get("custom_prompt") == target_prompt:
            return task

    # -----------------------------
    # 2. Normalized match (safe)
    # -----------------------------
    def normalize(text):
        return " ".join(str(text).strip().lower().split())

    target_norm = normalize(target_prompt)

    for task in tasks:
        if normalize(task.get("custom_prompt", "")) == target_norm:
            return task

    return None


# fetch_task_by_custom_prompt(
#     tasks,
#     "Analyze the provided source documents to determine the number of equipment models or variants.\n\nApply the following rules strictly.\n\nRule 1. If only one model or no variants are identified, write exactly: N/A - One Model\n\nRule 2. If more than one model or variant is identified, describe the differences between the models in exactly 100-200 words.",
# )

In [43]:
results = result[:10]

model_comparison_1 = []

for res in results:
    model_res = run_single_task_stats(task=res, rag_pipeline=rag_image)

    try:
        answer_json = json.loads(model_res["answer"])
        answer_text = answer_json.get("response", "")
    except Exception:
        answer_text = model_res.get("answer", "")

    model_comparison_1.append(
        {
            "completion_token": model_res["_token_usage"].get("completion"),
            "prompt_tokens": model_res["_token_usage"].get("prompt"),
            "total_token": model_res["_token_usage"].get("total"),
            "answer": answer_text,
            "model": "gpt-5.4",
            "query": model_res["metadata"]["question"]
        }
    )

Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6811
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6772
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6697
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7475
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7643
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7543
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6632
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6680
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6615
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6809


In [44]:
# --------------------------------------------------------
# Import Azure OpenAI and RAG utilities
# --------------------------------------------------------

from langchain_openai import AzureChatOpenAI
from langchain_community.callbacks import get_openai_callback

from utility.trf_report.trf_generation import (
    AOAI_ENDPOINT,
    AOAI_KEY,
    API_VERSION,
    CHAT_DEPLOY,
    build_rag_image_pipeline_grey,
    build_vision_message_grey,
    attach_supporting_refs_grey,
)

# --------------------------------------------------------
# Initialize Azure OpenAI LLM
# --------------------------------------------------------


def initialize_llm():
    """
    Create and configure the Azure OpenAI chat model.

    This LLM is used for structured AI extraction and response
    generation within the RAG processing pipeline.
    """
    print("Initializing Azure OpenAI model...")

    llm_instance = AzureChatOpenAI(
        azure_endpoint=AOAI_ENDPOINT,
        api_key=AOAI_KEY,
        openai_api_version="2025-01-01-preview",
        azure_deployment="gpt-4.1",
        temperature=0.1,
    ).with_config({"response_format": "json_object"})

    print("LLM initialized successfully")

    return llm_instance


# Create LLM instance
llm = initialize_llm()

# --------------------------------------------------------
# Build RAG image pipeline
# --------------------------------------------------------


def initialize_rag_pipeline(
    retriever_instance,
    llm_instance,
    vectorstore,
):
    """
    Build the multimodal RAG pipeline for task execution.

    This pipeline combines semantic retrieval, image context,
    and LLM reasoning to generate structured AI responses.
    """
    print("Building RAG image pipeline...")

    rag_pipeline = build_rag_image_pipeline_grey(
        retriever_instance,
        llm_instance,
        build_vision_message_grey,
        attach_supporting_refs_grey,
        vectorstore,
    )

    print("RAG image pipeline initialized")

    return rag_pipeline


# Create RAG pipeline
rag_image = initialize_rag_pipeline(
    retriever,
    llm,
    vs,
)

print(rag_image)

# --------------------------------------------------------
# Run single task with token statistics
# --------------------------------------------------------


def run_single_task_stats(task, rag_pipeline):
    """
    Execute a single AI task and capture token usage statistics.

    This function helps monitor prompt, completion, and total
    token consumption for performance and cost tracking.
    """
    print("Running task through RAG pipeline...")

    with get_openai_callback() as callback:
        response = rag_pipeline.invoke(task)

    # Attach token usage metadata
    response["_token_usage"] = {
        "prompt": callback.prompt_tokens,
        "completion": callback.completion_tokens,
        "total": callback.total_tokens,
    }

    print("Task completed successfully")
    print(f"Total tokens used: {callback.total_tokens}")

    return response

Initializing Azure OpenAI model...
LLM initialized successfully
Building RAG image pipeline...
RAG image pipeline initialized
first={
  retrieved_docs: RunnableLambda(itemgetter('question'))
                  | AzureCosmosDBNoSqlVectorStoreRetriever(vectorstore=<langchain_azure_ai.vectorstores.azure_cosmos_db_no_sql.AzureCosmosDBNoSqlVectorSearch object at 0x00000259235D4EC0>, search_kwargs={'with_embedding': False, 'offset_limit': None, 'projection_mapping': None, 'full_text_rank_filter': None, 'where': None, 'weights': None, 'score_threshold': 0.5}),
  question: RunnableLambda(itemgetter('question')),
  image: RunnableLambda(itemgetter('image')),
  task_type: RunnableLambda(itemgetter('task_type')),
  custom_prompt: RunnableLambda(itemgetter('custom_prompt')),
  accuracy_level: RunnableLambda(itemgetter('accuracy_level')),
  grey: RunnableLambda(itemgetter('grey'))
} middle=[{
  inputs: RunnableLambda(lambda x: {**x, 'context': extract_docs(x.get('retrieved_docs')), 'retrieved_docs':

In [46]:
result[0]

{'question': "Verify if the manufacturer mentioned a fluid anywhere in the documentation, the equipment must be safe when exposed to that fluid.Mention 'No fluid considered' incase there is no mention of any fluids by the manufacturer in the documentation.",
 'image': [{'pdf_file': 'ESP32-WROOM-32D_ESP-WROOM-32D_CE_Certification.pdf',
   'page': 2,
   'image_file': 'ESP32-WROOM-32D_ESP-WROOM-32D_CE_Certification.pdf_page_2.png',
   'url': 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/ESP32-WROOM-32D_ESP-WROOM-32D_CE_Certification.pdf/page_2.png'},
  {'pdf_file': 'ESP32-WROOM-32D_ESP-WROOM-32D_CE_Certification.pdf',
   'page': 3,
   'image_file': 'ESP32-WROOM-32D_ESP-WROOM-32D_CE_Certification.pdf_page_3.png',
   'url': 'https://stintertekesusdev.blob.core.windows.net/stintertekesusdev-blob/ESP32-WROOM-32D_ESP-WROOM-32D_CE_Certification.pdf/page_3.png'},
  {'pdf_file': 'Mornsun_5V_2A_regulator_K78xx-2000R3.pdf',
   'page': 2,
   'image_file': 'Mornsun_5V_2A_reg

In [47]:
new_res = []
for res in result:
    if res['accuracy_level'] == True:
        new_res.append(res)

In [ ]:
results = new_res[:10]

model_comparison_2 = []

for res in results:
    model_res = run_single_task_stats(task=res, rag_pipeline=rag_image)

    try:
        answer_json = json.loads(model_res["answer"])
        answer_text = answer_json.get("response", "")
    except Exception:
        answer_text = model_res.get("answer", "")

    model_comparison_2.append(
        {
            "completion_token": model_res["_token_usage"].get("completion"),
            "prompt_tokens": model_res["_token_usage"].get("prompt"),
            "total_token": model_res["_token_usage"].get("total"),
            "answer": answer_text,
            "model": "gpt-4.1",
            "query": model_res["metadata"]["question"]
        }
    )

Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7012
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7216
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6858
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7927
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 8060
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 8083
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 6781
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7187
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7043
Running task through RAG pipeline...
Task completed successfully
Total tokens used: 7235


In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity
from langchain_openai import AzureOpenAIEmbeddings

# ============================================================
# INPUT DATA
# ============================================================

data = model_comparison_1

# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(data)

# Strip answers
df["answer"] = df["answer"].astype(str).str.strip()

# Split by model
gpt_41 = df[df["model"] == "gpt-4.1"].reset_index(drop=True)
gpt_54 = df[df["model"] == "gpt-5.4"].reset_index(drop=True)

# ============================================================
# AZURE OPENAI EMBEDDINGS
# ============================================================

load_dotenv(override=True)
# Load environment variables
AOAI_ENDPOINT = os.getenv("aoai-endpoint")
AOAI_KEY = os.getenv("aoai-key")
API_VERSION = os.getenv("api-version")
EMBED_DEPLOY = os.getenv("embed-deploy")

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
    api_version=API_VERSION,
    azure_deployment=EMBED_DEPLOY,
)

# ============================================================
# BUILD COMPARISON TABLE
# ============================================================

comparison_rows = []

for idx in range(min(len(gpt_41), len(gpt_54))):

    ans_41 = str(gpt_41.loc[idx, "answer"])
    ans_54 = str(gpt_54.loc[idx, "answer"])

    # Generate embeddings
    emb_41 = embeddings.embed_query(ans_41)
    emb_54 = embeddings.embed_query(ans_54)

    # Cosine similarity
    similarity = cosine_similarity(
        [emb_41],
        [emb_54]
    )[0][0]

    comparison_rows.append(
        {
            "sr.no": idx + 1,

            "token_prompt_gpt_4.1":
                gpt_41.loc[idx, "prompt_tokens"],

            "token_prompt_gpt_5.4":
                gpt_54.loc[idx, "prompt_tokens"],

            "token_completion_gpt_4.1":
                gpt_41.loc[idx, "completion_token"],

            "token_completion_gpt_5.4":
                gpt_54.loc[idx, "completion_token"],

            "token_total_gpt_4.1":
                gpt_41.loc[idx, "total_token"],

            "token_total_gpt_5.4":
                gpt_54.loc[idx, "total_token"],
            "question":
                gpt_41.loc[idx, "prompt_tokens"],

            "answer_gpt_4.1":
                ans_41,

            "answer_gpt_5.4":
                ans_54,

            "cosine_similarity_score":
                round(float(similarity), 4),
        }
    )

# ============================================================
# FINAL DATAFRAME
# ============================================================

comparison_df = pd.DataFrame(comparison_rows)

# ============================================================
# DISPLAY
# ============================================================

print(comparison_df.to_markdown(index=False))

# ============================================================
# SAVE FILES
# ============================================================

comparison_df.to_csv(
    "gpt_model_comparison.csv",
    index=False
)

comparison_df.to_excel(
    "gpt_model_comparison.xlsx",
    index=False
)

print("\nComparison files saved successfully.")

|   sr.no |   token_prompt_gpt_4.1 |   token_prompt_gpt_5.4 |   token_completion_gpt_4.1 |   token_completion_gpt_5.4 |   token_total_gpt_4.1 |   token_total_gpt_5.4 | answer_gpt_4.1                                                                                                                                                                                                                                                                                                                              | answer_gpt_5.4                                  |   cosine_similarity_score |
|--------:|-----------------------:|-----------------------:|---------------------------:|---------------------------:|----------------------:|----------------------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [40]:
model_comparison_1

[{'completion_token': 86,
  'prompt_tokens': 6926,
  'total_token': 7012,
  'answer': 'Moisturizing nylon parts mentioned; fluid exposure considered.',
  'model': 'gpt-5.4',
  'query': "Verify if the manufacturer mentioned a fluid anywhere in the documentation, the equipment must be safe when exposed to that fluid.Mention 'No fluid considered' incase there is no mention of any fluids by the manufacturer in the documentation."},
 {'completion_token': 187,
  'prompt_tokens': 7144,
  'total_token': 7331,
  'answer': 'N/A',
  'model': 'gpt-5.4',
  'query': 'Verify if the manufacturer mentioned a fluid anywhere in the documentation, the equipment must be safe when exposed to that fluid.'},
 {'completion_token': 178,
  'prompt_tokens': 6680,
  'total_token': 6858,
  'answer': 'TBD - No info available',
  'model': 'gpt-5.4',
  'query': 'Verify incase there are battery electrolytes the leakage presents no hazard.'},
 {'completion_token': 162,
  'prompt_tokens': 7761,
  'total_token': 7923,
  '

In [50]:
# ============================================================
# MODEL COMPARISON FRAMEWORK
# Supports:
#   - gpt-4.1
#   - gpt-5.4
#   - gpt-5.4-mini
# Easily extensible for future models
# ============================================================

import os
import json
import pandas as pd

from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

from langchain_openai import (
    AzureChatOpenAI,
    AzureOpenAIEmbeddings,
)

from langchain_community.callbacks import get_openai_callback

from utility.trf_report.trf_generation import (
    AOAI_ENDPOINT,
    AOAI_KEY,
    API_VERSION,
    build_rag_image_pipeline_grey,
    build_vision_message_grey,
    attach_supporting_refs_grey,
)

# ============================================================
# ENV VARIABLES
# ============================================================

load_dotenv(override=True)

EMBED_DEPLOY = os.getenv("embed-deploy")

# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_DEPLOYMENTS = {
    "gpt-4.1": "gpt-4.1",
    "gpt-5.4": "gpt-5.4",
    "gpt-5.4-mini": "gpt-5.4-mini",
}

# ============================================================
# INITIALIZE LLM
# ============================================================


def initialize_llm(model_name):
    """
    Initialize Azure OpenAI chat model.
    """

    print(f"\nInitializing model: {model_name}")

    llm_instance = AzureChatOpenAI(
        azure_endpoint=AOAI_ENDPOINT,
        api_key=AOAI_KEY,
        openai_api_version="2025-01-01-preview",
        azure_deployment=MODEL_DEPLOYMENTS[model_name],
        temperature=0.1,
    ).with_config(
        {
            "response_format": "json_object"
        }
    )

    return llm_instance


# ============================================================
# INITIALIZE RAG PIPELINE
# ============================================================


def initialize_rag_pipeline(
    retriever_instance,
    llm_instance,
    vectorstore,
):
    """
    Build multimodal RAG pipeline.
    """

    rag_pipeline = build_rag_image_pipeline_grey(
        retriever_instance,
        llm_instance,
        build_vision_message_grey,
        attach_supporting_refs_grey,
        vectorstore,
    )

    return rag_pipeline


# ============================================================
# RUN SINGLE TASK
# ============================================================


def run_single_task_stats(
    task,
    rag_pipeline,
):
    """
    Run one task and capture token usage.
    """

    with get_openai_callback() as callback:

        response = rag_pipeline.invoke(task)

    response["_token_usage"] = {
        "prompt": callback.prompt_tokens,
        "completion": callback.completion_tokens,
        "total": callback.total_tokens,
    }

    return response


# ============================================================
# RUN MODEL ON TASKS
# ============================================================


def evaluate_model(
    model_name,
    tasks,
    retriever,
    vectorstore,
):
    """
    Evaluate one model across all tasks.
    """

    print(f"\nRunning evaluation for {model_name}")

    llm = initialize_llm(model_name)

    rag_pipeline = initialize_rag_pipeline(
        retriever,
        llm,
        vectorstore,
    )

    model_results = []

    for idx, task in enumerate(tasks):

        print(
            f"[{model_name}] Processing task "
            f"{idx + 1}/{len(tasks)}"
        )

        model_res = run_single_task_stats(
            task=task,
            rag_pipeline=rag_pipeline,
        )

        # --------------------------------------------
        # Parse answer safely
        # --------------------------------------------

        try:
            answer_json = json.loads(
                model_res["answer"]
            )

            answer_text = answer_json.get(
                "response",
                "",
            )

        except Exception:
            answer_text = model_res.get(
                "answer",
                "",
            )

        model_results.append(
            {
                "question":
                    model_res["metadata"]["question"],

                "prompt_tokens":
                    model_res["_token_usage"]["prompt"],

                "completion_tokens":
                    model_res["_token_usage"]["completion"],

                "total_tokens":
                    model_res["_token_usage"]["total"],

                "answer":
                    str(answer_text).strip(),

                "model":
                    model_name,
            }
        )

    return model_results


# ============================================================
# EMBEDDING MODEL
# ============================================================

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
    api_version=API_VERSION,
    azure_deployment=EMBED_DEPLOY,
)

# ============================================================
# MAIN COMPARISON FUNCTION
# ============================================================


def compare_models(
    tasks,
    model_names,
    retriever,
    vectorstore,
):
    """
    Compare multiple models on same tasks.
    """

    # --------------------------------------------
    # Run all models
    # --------------------------------------------

    all_results = {}

    for model_name in model_names:

        model_output = evaluate_model(
            model_name=model_name,
            tasks=tasks,
            retriever=retriever,
            vectorstore=vectorstore,
        )

        all_results[model_name] = model_output

    # --------------------------------------------
    # Build comparison rows
    # --------------------------------------------

    comparison_rows = []

    total_tasks = len(tasks)

    baseline_model = model_names[0]

    for idx in range(total_tasks):

        row = {
            "sr.no": idx + 1,
            "question":
                all_results[baseline_model][idx]["question"],
        }

        # ----------------------------------------
        # Add model-specific data
        # ----------------------------------------

        for model_name in model_names:

            result = all_results[model_name][idx]

            safe_model_name = (
                model_name.replace(".", "_")
                          .replace("-", "_")
            )

            row[
                f"token_prompt_{safe_model_name}"
            ] = result["prompt_tokens"]

            row[
                f"token_completion_{safe_model_name}"
            ] = result["completion_tokens"]

            row[
                f"token_total_{safe_model_name}"
            ] = result["total_tokens"]

            row[
                f"answer_{safe_model_name}"
            ] = result["answer"]

        # ----------------------------------------
        # Cosine similarity calculations
        # ----------------------------------------

        baseline_answer = all_results[
            baseline_model
        ][idx]["answer"]

        baseline_embedding = embeddings.embed_query(
            baseline_answer
        )

        for compare_model in model_names[1:]:

            compare_answer = all_results[
                compare_model
            ][idx]["answer"]

            compare_embedding = embeddings.embed_query(
                compare_answer
            )

            similarity = cosine_similarity(
                [baseline_embedding],
                [compare_embedding],
            )[0][0]

            safe_compare_name = (
                compare_model.replace(".", "_")
                             .replace("-", "_")
            )

            row[
                f"cosine_similarity_"
                f"{baseline_model}_vs_"
                f"{safe_compare_name}"
            ] = round(
                float(similarity),
                4,
            )

        comparison_rows.append(row)

    # --------------------------------------------
    # Final DataFrame
    # --------------------------------------------

    comparison_df = pd.DataFrame(
        comparison_rows
    )

    return comparison_df


# ============================================================
# FILTER TASKS
# ============================================================

filtered_tasks = [
    r for r in result
    if r.get("accuracy_level") is True
]

filtered_tasks = filtered_tasks[:10]

# ============================================================
# RUN COMPARISON
# ============================================================

comparison_df = compare_models(
    tasks=filtered_tasks,
    model_names=[
        "gpt-4.1",
        "gpt-5.4",
        "gpt-5.4-mini",
    ],
    retriever=retriever,
    vectorstore=vs,
)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n")
print(comparison_df.to_markdown(index=False))

# ============================================================
# SAVE RESULTS
# ============================================================

comparison_df.to_csv(
    "multi_model_comparison.csv",
    index=False,
)

comparison_df.to_excel(
    "multi_model_comparison.xlsx",
    index=False,
)

print("\nComparison completed successfully.")
print("Files saved:")
print(" - multi_model_comparison.csv")
print(" - multi_model_comparison.xlsx")


Running evaluation for gpt-4.1

Initializing model: gpt-4.1
[gpt-4.1] Processing task 1/10
[gpt-4.1] Processing task 2/10
[gpt-4.1] Processing task 3/10
[gpt-4.1] Processing task 4/10
[gpt-4.1] Processing task 5/10
[gpt-4.1] Processing task 6/10
[gpt-4.1] Processing task 7/10
[gpt-4.1] Processing task 8/10
[gpt-4.1] Processing task 9/10
[gpt-4.1] Processing task 10/10

Running evaluation for gpt-5.4

Initializing model: gpt-5.4
[gpt-5.4] Processing task 1/10
[gpt-5.4] Processing task 2/10
[gpt-5.4] Processing task 3/10
[gpt-5.4] Processing task 4/10
[gpt-5.4] Processing task 5/10
[gpt-5.4] Processing task 6/10
[gpt-5.4] Processing task 7/10
[gpt-5.4] Processing task 8/10
[gpt-5.4] Processing task 9/10
[gpt-5.4] Processing task 10/10

Running evaluation for gpt-5.4-mini

Initializing model: gpt-5.4-mini
[gpt-5.4-mini] Processing task 1/10
[gpt-5.4-mini] Processing task 2/10
[gpt-5.4-mini] Processing task 3/10
[gpt-5.4-mini] Processing task 4/10
[gpt-5.4-mini] Processing task 5/10
[gpt-